In [167]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2

from funkcje import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [189]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from optbinning import OptimalBinning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score

In [169]:
#Dane
df = pd.read_csv("zbior_1.csv")
#usuwam wybrane zmienne kategoryczne na podstawie tego co w eksploracja_danych.ipynb
X=df.iloc[:, :-1].drop(columns=["szczegolnaFormaPrawna_Symbol", "schemat_wsk_rzis","schemat_wsk_bilans"])
y=df.iloc[:, -1]

#Podzil na zbior treningowy (70%) i pozostale
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=2137,
    stratify=y
)

#Podzil na zbior walidacyjny (15%) i testowy (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=2137,
    stratify=y_temp
)

In [170]:
#Podzial na cechy numeryczne i kategoryczne
numeric_columns = X_train.select_dtypes(include=['float']).columns
numeric_columns = numeric_columns.append(pd.Index(['wsk_liczba_dni_istnienia']))
categorical_columns = X_train.select_dtypes(include=['str', 'int']).columns
categorical_columns = categorical_columns.drop('wsk_liczba_dni_istnienia')

X_cat=X_train[categorical_columns]
X_num=X_train[numeric_columns]

#zastopinie nieskonczonosci przez Nan
X_num = X_num.replace([np.inf, -np.inf], np.nan)

In [171]:
mask = (X_num.eq(0) | X_num.isnull()).mean() <= 0.5
X_num = X_num.loc[:, mask]

In [172]:
#usnicie kolumn o kretnej korelacji powyzej 0.95
X_num =usuwanie_najbardziej_scorrelowanych(X_num, prog=0.95)

Usunięto 69 kolumn.


In [173]:
#policzenie jednowymiarowego AUC dla każdej kolumny numerycznej, Małysz twierdził, że to pomaga zobaczyć jak cech jest ważna
cechy_num = [c for c in X_num.columns]
rows = []
for c in cechy_num:

    m = X_num[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_num.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
wsk_index_skr_zaw,0.700,+,0.000
wsk_index_nowe,0.694,+,0.000
wsk_marza_brutto_1,0.663,-,0.000
wsk_ebitda_zobowiazan_1,0.657,-,0.000
wsk_ebitda_zobowiazan_2,0.656,-,0.000
wsk_ROE_brutto,0.656,-,0.000
wsk_ebitda_zobowiazan_odsetki_2,0.654,-,0.000
wsk_s_ROE,0.654,-,0.000
wsk_marza_ebitda_1,0.654,-,0.000


Podejrzanie silne cechy (AUC > 0,85): []


In [174]:
#usnieceie kolumn o AUC niższym niż 0.6 i pomnozenie wierszy z "+" przez -1
kolumny_do_usuniecia = tab_auc[tab_auc["AUC jednowymiarowe"] < 0.6].index.tolist()
mask_odwroc = (tab_auc["kierunek"] == "+") & (tab_auc["AUC jednowymiarowe"] >= 0.6)
kolumny_do_odwrocenia = tab_auc[mask_odwroc].index.tolist()

X_num = X_num.drop(columns=kolumny_do_usuniecia)
X_num[kolumny_do_odwrocenia] = -X_num[kolumny_do_odwrocenia]

In [175]:
#policzenie jednowymiarowego AUC dla każdej kolumny numerycznej jeszcze raz
cechy_num = [c for c in X_num.columns]
rows = []
for c in cechy_num:

    m = X_num[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_num.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
wsk_index_skr_zaw,0.700,-,0.000
wsk_index_nowe,0.694,-,0.000
wsk_marza_brutto_1,0.663,-,0.000
wsk_ebitda_zobowiazan_1,0.657,-,0.000
wsk_ebitda_zobowiazan_2,0.656,-,0.000
wsk_ROE_brutto,0.656,-,0.000
wsk_ebitda_zobowiazan_odsetki_2,0.654,-,0.000
wsk_s_ROE,0.654,-,0.000
wsk_marza_ebitda_1,0.654,-,0.000


Podejrzanie silne cechy (AUC > 0,85): []


In [176]:
#zmiana zmiennych kategorycznych na numeryczne przy pomocy WoE z optbinning
for col in X_cat.columns:
    optb = OptimalBinning(
        name=col,
        dtype="categorical",
        solver="cp",
        cat_cutoff=0.01,
        max_n_bins=4
    )

    optb.fit(X_cat[col], y_train)

    X_cat[col] = optb.transform(
        X_cat[col], metric="woe"
    )
    X_valid[col] = optb.transform(
        X_valid[col], metric="woe"
    )



In [180]:
#znalezienie kolumn, do przeskalowania
skosne_cechy = X_num.skew().sort_values(ascending=False)
do_przeskalowania = skosne_cechy[(skosne_cechy > 2) & (X_num.min() >= 0)]
print("Cechy do przeskalowania:", do_przeskalowania.index.tolist())

Cechy do przeskalowania: ['Aktywa']


In [181]:
#zlogrytmowanie cech o skośności większej niż 2
for col in do_przeskalowania.index.tolist():
    X_num[col] = np.log1p(X_num[col])
    X_valid[col] = np.log1p(X_valid[col])
#polaczenie cech numerycznych i kategorycznych w jeden zbiór
X_train_final = pd.concat([X_num, X_cat], axis=1)

In [182]:
#napisanie pipeline, który będzie przeskalowywał cechy i inputował nany i trenował regresję logistyczną
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', LogisticRegression())
])

In [187]:
X_valid = X_valid[X_train_final.columns]
X_valid = X_valid.replace([np.inf, -np.inf], np.nan)

In [191]:
pipeline.fit(X_train_final, y_train)
pred=pipeline.predict(X_valid)

In [192]:
#policzmy przeskalowane acc na predykcjach
balanced_accuracy_score(y_valid, pred)

0.5088414634146341